# 02 — Preprocessing

**Goal:** produce `data/processed/train.parquet` and `test.parquet` ready for Phase 6 model training, with the label encoder persisted to `models/label_encoder.joblib`.

**Critical rule (ADR-006):** feature scaling does NOT happen here. It lives inside the sklearn Pipeline so it refits per CV fold — that's the structural defence against data leakage. Read `docs/ml_pipeline.md` if this isn't obvious.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.logging import configure_logging
configure_logging(level='INFO')

from src.config.constants import (
    LABEL_COLUMN, MODELS_DIR, PROCESSED_DIR, RANDOM_STATE,
    SAMPLE_DIR, TARGET_LABELS,
)

## Step 1 — Load + clean + filter

Identical to notebook 01 up to this point. The cleaning step is idempotent so re-running is safe.

In [ ]:
from src.data.loader import load_raw
from src.features.cleaning import clean, filter_target_classes

df = load_raw(raw_dir=SAMPLE_DIR)
df = clean(df)
df = filter_target_classes(df, TARGET_LABELS)
print('After cleaning:', df.shape)

## Step 2 — Label encoding

`fit_label_encoder` checks that the fitted class set equals `TARGET_LABELS`. If it raises, the most likely cause is that `filter_target_classes` was skipped and an extra label leaked in.

In [ ]:
from src.features.encoder import fit_label_encoder, save_label_encoder
import pandas as pd

le = fit_label_encoder(df[LABEL_COLUMN])
y = pd.Series(le.transform(df[LABEL_COLUMN]), index=df.index, name='label_encoded')
X = df.drop(columns=[LABEL_COLUMN])

print('Label mapping:')
for cls, idx in zip(le.classes_, le.transform(le.classes_)):
    print(f'  {idx} -> {cls}')

## Step 3 — Stratified train/test split

`stratify=y` is non-negotiable — without it the minority classes (FTP-Patator most of all) can vanish from the test set entirely, ruining per-class recall.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE,
)
print('Train:', X_train.shape, '  Test:', X_test.shape)
print('\nTrain class balance:')
print(y_train.value_counts(normalize=True).round(3))
print('\nTest class balance:')
print(y_test.value_counts(normalize=True).round(3))

## Step 4 — Persist artifacts

Parquet, not CSV: 10× smaller, dtypes preserved, fast to read back in Phase 6.

The label encoder is saved with joblib so the dashboard's Predict-New-CSV page can decode predictions back to class names.

In [ ]:
import json

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

train_path = PROCESSED_DIR / 'train.parquet'
test_path = PROCESSED_DIR / 'test.parquet'

pd.concat([X_train, y_train], axis=1).to_parquet(train_path, index=False)
pd.concat([X_test, y_test], axis=1).to_parquet(test_path, index=False)
save_label_encoder(le)

(PROCESSED_DIR / 'feature_names.json').write_text(
    json.dumps(list(X.columns), indent=2), encoding='utf-8',
)

print('Wrote:')
print(' ', train_path)
print(' ', test_path)
print(' ', MODELS_DIR / 'label_encoder.joblib')

## Sanity checks

Load the artifacts back and confirm they round-trip.

In [ ]:
from src.features.encoder import load_label_encoder

train_back = pd.read_parquet(train_path)
test_back = pd.read_parquet(test_path)
le_back = load_label_encoder()

print('train round-trip ok:', train_back.shape == (len(X_train), X.shape[1] + 1))
print('test round-trip ok :', test_back.shape == (len(X_test), X.shape[1] + 1))
print('encoder classes ok :', le_back.classes_.tolist() == le.classes_.tolist())